# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para las celdas Markdown.

Material educativo sujeto a revisión y validación en Microsoft Fabric.

# NB_06 — Corrección gramatical con `ai.fix_grammar`

**Serie:** Microsoft Fabric AI Functions con Pandas

**Modelo:** `gpt-5-mini`

## 1. Objetivos y requisitos
Corregir ortografía, gramática y puntuación por fila, conservar significado y términos técnicos, validar nulos y medir tiempo, memoria, tokens y CU-segundos. Requiere capacidad Fabric compatible, permisos y runtime Python con AI Functions.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, e6352c52-f895-4a7b-8027-e7c5e3f0295d, 3, Finished, Available, Finished, False)

## 2. Configuración
Modelo gpt-5-mini, concurrencia de laboratorio 10 y barra de progreso con estadísticas. [Configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration).

In [2]:
import pandas as pd
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "gpt-5-mini"
CONCURRENCY = 10
aifunc.default_conf.model_deployment_name = MODEL_NAME
aifunc.default_conf.reasoning_effort = "low"
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"

StatementMeta(, e6352c52-f895-4a7b-8027-e7c5e3f0295d, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — ai.fix_grammar con pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/fix-grammar)

```python
df["corrections"] = df["input"].ai.fix_grammar()
```

| Parámetro | Tipo | Uso |
|---|---|---|
| Ninguno | — | Corrección estándar de texto. |
| `column_type` | str, opcional | `path` para archivos PDF o texto, según requisitos multimodales. |

Devuelve una Series con texto corregido por fila; una entrada nula produce nulo. No requiere idioma de destino. Los ejercicios siguientes son propios y deben revisarse manualmente.

## 4. Dataset ficticio
Se conservan los siete registros del NB_06 PySpark: errores intencionales en español e inglés, una frase correcta y un nulo.

In [3]:
data = [
    (1, "Los servidor no responde desde ayer"),
    (2, "El equipo de soporte a solucionado el problema."),
    (3, "Necesitamos revisar la configuracion del almacen de datos"),
    (4, "The reports was not updated yesterday."),
    (5, "We has a problem with the database connexion."),
    (6, "La carga de datos finalizó correctamente."),
    (7, None),
]
df = pd.DataFrame(data, columns=["id", "texto"])
display(df)

StatementMeta(, e6352c52-f895-4a7b-8027-e7c5e3f0295d, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2cba5aba-62a8-427d-a413-1b765c4e76df)

## 5. Ejecutar `ai.fix_grammar`
Conservar la Series original y sus estadísticas antes de asignarla al DataFrame. Medir tiempo y memoria aproximada de la entrada.

In [4]:
runs = []
def ejecutar(nombre, entrada, **kwargs):
    run_id = str(uuid.uuid4())
    started = datetime.now(timezone.utc).isoformat()
    input_bytes = int(entrada.memory_usage(deep=True).sum() if isinstance(entrada, pd.DataFrame) else entrada.memory_usage(deep=True))
    t0 = perf_counter()
    resultado = entrada.ai.fix_grammar(**kwargs)
    elapsed = perf_counter() - t0
    raw = resultado.ai.stats
    runs.append((nombre, run_id, started, elapsed, raw, resultado, len(entrada), input_bytes))
    return resultado


correccion = ejecutar("texto", df["texto"])
display(df.assign(correccion=correccion))

StatementMeta(, e6352c52-f895-4a7b-8027-e7c5e3f0295d, 6, Finished, Available, Finished, False)

ai.fix_grammar: 100%|██████████| 7/7 [00:02<00:00,  3.17it/s, cached=0, in=956, out=618, CU-s=42.35]                            


SynapseWidget(Synapse.DataFrame, 9e879bbc-b546-461e-a6a8-3194d274839e)

## 6. Evaluar cambios
Una frase modificada no implica mejora. Revisar concordancia, acentuación, puntuación y conservación del significado. La frase correcta sirve como control; no se requiere una segunda inferencia.

In [5]:
resultado = df.assign(correccion=correccion)
estado = pd.Series("MODIFICADO", index=df.index)
estado.loc[df.texto.eq(correccion).fillna(False)] = "SIN_CAMBIOS"
estado.loc[correccion.isna() & df.texto.notna()] = "SIN_RESULTADO"
estado.loc[df.texto.isna()] = "ENTRADA_NULA"
resultado["estado"] = estado
display(resultado)
display(estado.value_counts(dropna=False))

StatementMeta(, e6352c52-f895-4a7b-8027-e7c5e3f0295d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 220b3322-4ff7-43c5-a6ce-d71b5dfb443b)

SynapseWidget(Synapse.DataFrame, 2cc1ba1c-8fcf-4410-8de7-c915688dab15)

## 7. Validación lingüística
Las referencias humanas siguientes son ejemplos, no salidas reales. Se admite más de una corrección válida. La coincidencia exacta no equivale a calidad semántica.

In [6]:
referencias = pd.Series([
    "Los servidores no responden desde ayer.",
    "El equipo de soporte ha solucionado el problema.",
    "Necesitamos revisar la configuración del almacén de datos.",
    "The reports were not updated yesterday.",
    "We have a problem with the database connection.",
    "La carga de datos finalizó correctamente.",
    None
], index=df.index)
resultado["referencia_humana"] = referencias
resultado["coincidencia_exacta"] = correccion.eq(referencias).where(referencias.notna(), pd.NA).astype("boolean")
display(resultado)
print("Salidas inesperadas para entradas nulas:", int((df.texto.isna() & correccion.notna()).sum()))

StatementMeta(, e6352c52-f895-4a7b-8027-e7c5e3f0295d, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d854e4b6-666a-4ba1-b66f-436d4b24d93d)

Salidas inesperadas para entradas nulas: 0


## 8. Estadísticas y tokens
Consultar cada ejecución por separado. No reemplazar métricas ausentes por cero.

In [7]:
def normalizar(raw):
    if isinstance(raw, pd.DataFrame) and len(raw)==1:
        return raw.iloc[0].to_dict()
    if isinstance(raw, pd.Series):
        return raw.to_dict()
    if isinstance(raw, dict):
        return raw
    raise RuntimeError("Revisar estructura de ai.stats")
for nombre, _, _, _, raw, _, _, _ in runs:
    print(nombre)
    display(raw)

StatementMeta(, e6352c52-f895-4a7b-8027-e7c5e3f0295d, 9, Finished, Available, Finished, False)

texto


SynapseWidget(Synapse.DataFrame, 8eeb9719-14f1-4819-a40a-9290a643d27e)

## 9. Estimación de CU-segundos
Tarifas de referencia de la serie para gpt-5-mini integrado, en CU-s por 1.000 tokens:

| Entrada sin caché | Entrada en caché | Salida |
|---:|---:|---:|
| 8,40 | 0,84 | 67,23 |

$CU_s=((I-C)\times8.40+C\times0.84+O\times67.23)/1000$

C es un subconjunto de I. Los tokens de razonamiento no se suman nuevamente a O. Ejemplo hipotético: I=1000, C=200, O=100 da 13,611 CU-s. No representa una ejecución real ni el consumo total del runtime. Verificar vigencia en [facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing). Si faltan métricas o el modelo/cliente no coincide, no estimar.

## 10. Auditoría individual
Registrar la inferencia principal sin guardar los textos. Las métricas ausentes permanecen nulas. El estado es técnico, no lingüístico.

In [8]:
records = []
for nombre, run_id, started, elapsed, raw, resultado_run, num_rows, input_bytes in runs:
    stats = normalizar(raw)
    def metric(k):
        v = stats.get(k)
        return None if v is None or pd.isna(v) else int(v)
    I,C,O = [metric(k) for k in ("input_tokens","cached_tokens","output_tokens")]
    cu = None
    if (stats.get("model") in (MODEL_NAME,"gpt-5-mini-2025-08-07") and stats.get("client_type")=="fabric_llm_endpoint" and all(v is not None for v in (I,C,O)) and 0<=C<=I and O>=0):
        cu = ((I-C)*8.40+C*0.84+O*67.23)/1000
    audit = dict(run_id=run_id, execution_timestamp_utc=started, notebook_name="NB_06_AI_FIX_GRAMMAR_PANDAS", operation="ai.fix_grammar", model_configured=MODEL_NAME, model_reported=stats.get("model"), num_input_rows=num_rows, client_type=stats.get("client_type"), estimated_cu_seconds=cu, elapsed_seconds=elapsed, input_bytes=input_bytes, output_bytes=int(resultado_run.memory_usage(deep=True)), concurrency=CONCURRENCY)
    for key in ("num_successful","num_exceptions","num_unevaluated","num_harmful","input_tokens","cached_tokens","output_tokens","reasoning_tokens"):
        audit[key]=metric(key)
    expected = int(df.texto.notna().sum())
    if (audit["num_exceptions"] or 0) > 0 or (audit["num_harmful"] or 0) > 0:
        audit["execution_status"] = "PARTIAL"
    elif audit["num_successful"] == expected and audit["num_exceptions"] == 0 and audit["num_harmful"] == 0 and resultado_run[df.texto.notna()].notna().all():
        audit["execution_status"] = "SUCCESS"
    else:
        audit["execution_status"] = "INCOMPLETE"
    records.append(audit)
audit_df=pd.DataFrame(records)
display(audit_df)
print("CU-s acumulados:", audit_df.estimated_cu_seconds.sum(min_count=len(audit_df)))

StatementMeta(, e6352c52-f895-4a7b-8027-e7c5e3f0295d, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, da3115bb-7be0-43f2-922f-db0b86e870ec)

CU-s acumulados: 49.578540000000004


## 11. Persistencia opcional en Delta
Requiere Lakehouse adjunto, deltalake y pyarrow. Persiste en Files, no registra una tabla SQL. Deshabilitada por defecto.

In [ ]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

## 12. Multimodalidad (opcional)
Usar archivos autorizados y revisar [requisitos multimodales](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview). Consumo independiente, fuera de la auditoría principal.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    archivos = pd.Series(["/lakehouse/default/Files/ejemplo.pdf"])
    resultado_archivos = archivos.ai.fix_grammar(column_type="path")
    display(resultado_archivos)
    display(resultado_archivos.ai.stats)

## 13. Comparación y conclusiones

| Pandas | PySpark |
|---|---|
| `df["texto"].ai.fix_grammar()` | `df.ai.fix_grammar(input_col="texto", output_col="correccion", error_col="error_correccion")` |
| Devuelve Series | Devuelve Spark DataFrame |
| Memoria local | Procesamiento distribuido |

Se conserva el dataset para comparar ambas implementaciones. Revisar manualmente fidelidad, términos técnicos y frases que ya eran correctas. El consumo estimado corresponde a inferencias, no al runtime completo.

- [Método oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/fix-grammar)
- [Configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)

Validación de ejecución pendiente en Fabric; verificar vigencia de tarifas.